Now that you've deployed your endpoint - it's time to slam it!

In [1]:
import os
import getpass

os.environ["TOGETHER_API_KEY"] = getpass.getpass("Enter your Together API key: ")

Enter your Together API key:  ········


Let's try with 1 request, just to verify our endpoint is alive.

Make sure you provide your own endpoint identifier! It will look something like this:

- `your-username-here/openai/gpt-oss-20b-unique-identifier`

In [7]:
from together import Together

client = Together()

# REPLACE WITH YOUR OWN ENDPOINT IDENTIFIER or sesrverless identity from together ai
model_endpoint = "openai/gpt-oss-20b"
response = client.chat.completions.create(
    model=model_endpoint,
    messages=[
      {
        "role": "user",
        "content": "How much wood could a wood chuck chuck if a wood chuck could chuck wood?"
      }
    ]
)
print(response.choices[0].message.content)


**Short answer:**  
About **700 pounds** of wood – if a woodchuck could actually chuck wood!

---

### The “700‑pound” estimate

In 1988 wildlife biologist Richard Thomas, in a whimsical column for the *New York Times*, ran a quick calculation:  
- A woodchuck (also known as a ground‑hog) can move roughly **35 pounds** of dirt in one burrow‑digging session.  
- A standard “wood‑chuck” would presumably move the same volume of wood.  

Multiplying the 35 pounds by **20** (one roughly estimates a woodchuck might dig about 20 burrows per season) yields about **700 pounds**.  

That number has since become the go‑to “scientific” answer to the tongue‑twister.

### A better (or at least more accurate) answer

- **Woodchucks don’t actually chuck wood.** They’re burrowing animals, not lumberjacks.  
- In the wild, they’re known to move soil (and occasionally small sticks) while building complex underground tunnels.  
- If you really had to ask: How much wood *could* a woodchuck physically move?

Now, let's SLAM IT.

In [3]:
import asyncio
from together import AsyncTogether

async def send_request(client, idx):
    try:
        response = await client.chat.completions.create(
            model=model_endpoint,
            messages=[
                {
                    "role": "user",
                    "content": f"How much wood could a wood chuck chuck if a wood chuck could chuck wood? (Request {idx})"
                }
            ]
        )
        print(f"Response {idx}: {response.choices[0].message.content[:60]}...")
    except Exception as e:
        print(f"Request {idx} failed: {e}")

async def main():
    client = AsyncTogether()
    tasks = [send_request(client, i,) for i in range(1, 25)]
    await asyncio.gather(*tasks)

# Run the async main function
await main()

Response 17: **Answer:**  
In folklore and popular culture, it’s often sa...
Response 7: A humorous take on that classic tongue‑twister is that, if a...
Response 5: wood chuck.

I am not a wood chuck, I am just a text AI.

th...
Response 11: **The classic tongue‑twister answer:**

> *“If a woodchuck c...
Response 14: It’s all in good fun—there’s no scientific basis for a real ...
Response 1: **Answer to “Request 1”**

The classic tongue‑twister goes: ...
Response 23: **Answer to “How much wood could a wood‑chuck chuck if a woo...
Response 22: **Answer (Request 22):**

> “A wood‑chuck would chuck as muc...
Response 9: **The classic answer (tongue‑twister style)**  
> “A wood‑ch...
Response 16: The old tongue‑twister goes: “How much wood would a wood‑chu...
Response 13: You’re tackling one of the most‑iconic tongue‑twisters in th...
Response 6: ### The classic answer

> **“A wood‐chuck would chuck as muc...
Response 19: There’s no hard‑science answer to the classic tongue‑twister...
Resp

### Using RAG from session 14. simple search using Tavily agent 

In [2]:
os.environ["TAVILY_API_KEY"]=getpass.getpass("Enter your Tavily API key:..")

Enter your Tavily API key:.. ········


In [8]:
# Test Simple Agent with Together AI Serverless Endpoint (openai/gpt-oss-20b)
# The agent can use RAG tool to search data/howpeopleuseai.pdf

from app.graphs.simple_agent import graph
from langchain_core.messages import HumanMessage

# Ensure TOGETHER_API_KEY is set (should already be set from first cell)
# The agent will automatically use openai/gpt-oss-20b via app/models.py

# Test query - agent can use RAG tool to search data/howpeopleuseai.pdf
print("Invoking simple agent with Together AI...")
print("Model: openai/gpt-oss-20b (serverless)")
print("=" * 60)

result = graph.invoke({
    "messages": [
        HumanMessage(content="What are some ways people are using AI in their daily work?")
    ]
})

# Print the final response
print("\nAgent Response:")
print("=" * 60)
print(result["messages"][-1].content)

Invoking simple agent with Together AI...
Model: openai/gpt-oss-20b (serverless)

Agent Response:
analysisThe retrieve_information tool failed due to token limit. We can ignore it; we already have answer. Provide final answer.assistantfinal**How people are weaving AI into their everyday work**

| # | Domain | Typical AI Tasks | Why it matters |
|---|--------|------------------|----------------|
| 1 | **Email & Calendar** | • Draft replies, summarize threads<br>• Auto‑prioritize inbox, schedule meetings | Cuts hours spent on routine communication and keeps schedules on track. |
| 2 | **Content Creation** | • Write blog posts, social‑media copy, newsletters<br>• Generate graphics, video scripts, design mock‑ups | Speeds up creative output and keeps content fresh. |
| 3 | **Data Analysis & Reporting** | • Clean datasets, spot outliers<br>• Auto‑generate charts, dashboards, executive summaries | Turns raw numbers into actionable insights with minimal manual effort. |
| 4 | **Customer Suppo

### Using simple agent with concurrency

In [9]:
# Simple concurrent requests to agent - shorter version
import asyncio
from app.graphs.simple_agent import graph
from langchain_core.messages import HumanMessage

# Define async request function
async def agent_request(idx):
    """Make a single async request to the agent."""
    try:
        result = await graph.ainvoke({
            "messages": [HumanMessage(content="how people may use ai in one sentence")]
        })
        response = result["messages"][-1].content
        print(f"Request {idx}: {response[:80]}...")
        return response
    except Exception as e:
        print(f"Request {idx} failed: {e}")
        return None

# Run 5 concurrent requests
print("Running 3 concurrent requests...")
print("=" * 60)
results = await asyncio.gather(*[agent_request(i) for i in range(1, 4)])
print(f"\nCompleted: {sum(1 for r in results if r)}/3 successful")

Running 3 concurrent requests...
Request 1: analysisUser asks: "how people may use ai in one sentence". They want a one-sent...
Request 2: analysisUser asks: "how people may use ai in one sentence". They want a one-sent...
Request 3: analysisUser asks: "how people may use ai in one sentence". They want a one-sent...

Completed: 3/3 successful
